# Práctica 4 · Comparación y evaluación final

Objetivo: comparar tres algoritmos sobre Wine, seleccionar con validación y consultar prueba una vez. Esta celda final cierra el experimento. No modifica retrospectivamente las prácticas anteriores.

La selección usa mayor F1 macro. Empates se resuelven a favor del primer candidato en el orden declarado. Las configuraciones están fijadas antes de prueba. No es una búsqueda exhaustiva de hiperparámetros.

**Fuentes:** Géron, 3.ª ed., caps. 2–5; Grus, 2.ª ed., caps. 13–16; Gallatin y Albon, Cookbook adjunto, caps. 3–6 (preprocesamiento). Las consignas y los datos sintéticos son elaboración didáctica. Véase `../apuntes.md` y `../datos/README.md`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
ROOT = Path.cwd()
if not (ROOT / 'datos').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'datos').is_dir():
    raise FileNotFoundError('Abra el notebook desde la carpeta del subtema o desde notebooks, conservando datos/.')
print('Versiones:', sklearn.__version__, np.__version__, pd.__version__)

Versiones: 1.8.0 2.3.5 2.2.3


In [2]:
df = pd.read_csv(ROOT / 'datos/wine.csv')
features = [c for c in df.columns if c not in ['clase', 'particion']]
train = df[df.particion == 'entrenamiento']
val = df[df.particion == 'validacion']
Xtr, ytr = train[features], train.clase
Xv, yv = val[features], val.clase
print('Entrenamiento:', Xtr.shape, 'Validación:', Xv.shape)

Entrenamiento: (106, 13) Validación: (36, 13)


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
candidates={
 'Logística':make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=3000,random_state=43)),
 'GaussianNB':make_pipeline(StandardScaler(),GaussianNB()),
 'SVM RBF':make_pipeline(StandardScaler(),SVC(kernel='rbf',C=1,gamma='scale'))}
rows=[]
for name,model in candidates.items():
    model.fit(Xtr,ytr);pr=model.predict(Xv)
    rows.append({'modelo':name,'exactitud_val':accuracy_score(yv,pr),'F1_macro_val':f1_score(yv,pr,average='macro')})
report=pd.DataFrame(rows);print(report.round(4).to_string(index=False))
base=DummyClassifier(strategy='most_frequent').fit(Xtr,ytr)
print('Referencia: F1 macro validación =',f1_score(yv,base.predict(Xv),average='macro'))
selected=report.loc[report.F1_macro_val.idxmax(),'modelo']
print('Seleccionado según criterio predefinido:',selected)

    modelo  exactitud_val  F1_macro_val
 Logística         0.9722        0.9740
GaussianNB         0.9444        0.9475
   SVM RBF         0.9444        0.9457
Referencia: F1 macro validación = 0.18666666666666668
Seleccionado según criterio predefinido: Logística


## Cierre: prueba reservada
El candidato ya está seleccionado. Reajustamos su pipeline con entrenamiento y validación combinados. La referencia aprende de los mismos datos. No se usan resultados de prueba para volver a seleccionar.

In [4]:
dev=df[df.particion!='prueba'];test=df[df.particion=='prueba']
final=candidates[selected];final.fit(dev[features],dev.clase)
pr=final.predict(test[features])
print('Resultado final del candidato:',selected)
print(classification_report(test.clase,pr,zero_division=0,digits=4))
print('Matriz final:\n',confusion_matrix(test.clase,pr,labels=[0,1,2]))
base_final=DummyClassifier(strategy='most_frequent').fit(dev[features],dev.clase)
print('F1 macro referencia en prueba:',f1_score(test.clase,base_final.predict(test[features]),average='macro'))
print('F1 macro seleccionado en prueba:',f1_score(test.clase,pr,average='macro'))

Resultado final del candidato: Logística
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        12
           1     1.0000    0.8571    0.9231        14
           2     0.8333    1.0000    0.9091        10

    accuracy                         0.9444        36
   macro avg     0.9444    0.9524    0.9441        36
weighted avg     0.9537    0.9444    0.9448        36

Matriz final:
 [[12  0  0]
 [ 0 12  2]
 [ 0  0 10]]
F1 macro referencia en prueba: 0.18666666666666668
F1 macro seleccionado en prueba: 0.9440559440559441


**Reporte final (1–2 páginas):** describa datos, particiones, pipelines, criterio previo, tabla de validación y resultado final. Interprete una confusión por clase y explique límites por tamaño de prueba. No extrapole el resultado a fallas de equipos o audio apícola.

**Extensión P4-A:** proponga cómo introducir validación cruzada en 4.5 sin reutilizar prueba para ajustar. Si realiza un experimento nuevo después de ver prueba, declare su carácter exploratorio y redefina una evaluación independiente.